In [0]:
# spark.sql("create schema sales_project_streaming.brz;")
# spark.sql("create schema sales_project_streaming.slv;")
# spark.sql("create schema sales_project_streaming.gld;")

In [0]:
# %sql
# create volume sales_project_streaming.slv.checkpoints_vol;

In [0]:
topics = {"sales": "sales_stream", 
          "employees": "employees_stream", 
          "expenses": "expenses_stream", 
          "regions": "regions_stream"}

checkpoints_brz = {"sales": "/Workspace/Shared/checkpoints_streaming/sales_chck_brz",
               "employees": "/Workspace/Shared/checkpoints_streaming/employees_chck_brz",
               "expenses": "/Workspace/Shared/checkpoints_streaming/expenses_chck_brz",
               "regions": "/Workspace/Shared/checkpoints_streaming/regions_chck_brz"}

checkpoints_slv = {"sales": "/Volumes/sales_project_streaming/slv/checkpoints_vol/sales_chck_slv_v2/",
               "employees": "/Volumes/sales_project_streaming/slv/checkpoints_vol/employees_chck_slv/",
               "expenses": "/Volumes/sales_project_streaming/slv/checkpoints_vol/expenses_chck_slv/",
               "regions": "/Volumes/sales_project_streaming/slv/checkpoints_vol/regions_chck_Slv/"}

ngrokip = "0.tcp.in.ngrok.io:15212"

In [0]:
# %sql
# create or replace table sales_project_streaming.brz.sales(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

# create or replace table sales_project_streaming.brz.employees(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

# create or replace table sales_project_streaming.brz.expenses(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

# create or replace table sales_project_streaming.brz.regions(
#     key string,
#   value string,
#   topic string,
#   partition int,
#   offset bigint,
#   timestamp timestamp,
#   ingestion_ts timestamp
# )
# using delta;

In [0]:
from pyspark.sql import functions as F

def consume_data(topic_key):

      df = (spark.readStream.format("kafka")
            .option("kafka.bootstrap.servers", ngrokip)
            .option("subscribe", topics[topic_key])
            .option("startingOffsets", "earliest")
            .option("failOnDataLoss", "false")
            .load())

      df = df.selectExpr("cast(key as string) as key",
                        "cast(value as string) as value",
                        "topic",
                        "partition",
                        "offset",
                        "timestamp").withColumn("ingestion_ts", F.current_timestamp())

      query = (df.writeStream.format("delta")
            .option("checkpointLocation", checkpoints_brz[topic_key])
            .trigger(availableNow = True)
            .outputMode("append")
            .table(f"sales_project_streaming.brz.{topic_key}"))

      return query

queries = {}

for topic_key, topic_value in topics.items():

      q = consume_data(topic_key)
      queries[topic_key] = q

for topic_key, q in queries.items():

      try:
            q.awaitTermination()
            print(f"Completed Stream for topic: {topics[topic_key]}")

      except Exception as e:
            print(f"error in topic {topics[topic_key]}: {e}")

In [0]:
# from pyspark.sql.types import *

# df = (spark.readStream
#       .format("delta")
#       .table("sales_project_streaming.brz.sales")
#       )

# schema = StructType([
#     StructField("sales_id", LongType(), False),
#     StructField("employee_id", LongType(), True),
#     StructField("region_id", IntegerType(), True),
#     StructField("product_id", IntegerType(), True),
#     StructField("quantity", IntegerType(), True),
#     StructField("sales_amount", LongType(), False),
#     StructField("event_time", TimestampType(), False),
#     StructField("ingestion_time", TimestampType(), False)
# ])

# parsed_df = df.withColumn("parsed_value", F.from_json("value", schema)).select("parsed_value", "ingestion_ts")

# normalised_df = parsed_df.select("parsed_value.sales_id", "parsed_value.employee_id", "parsed_value.region_id",
#                                  "parsed_value.product_id", "parsed_value.quantity", 
#                                  F.col("parsed_value.sales_amount").alias("amount"),
#                                  "parsed_value.event_time")

# normalised_df = (normalised_df.withWatermark("event_time", "10 minutes").dropDuplicates(subset = ["sales_id"])
#                  .withColumn("hour", F.hour(F.col("event_time")))
#                  .withColumn("event_date", F.to_date(F.col("event_time")))
#                  .withColumn("day", F.date_format(F.col("event_time"), "EEEE"))
#                  .withColumn("day_of_week", F.dayofweek(F.col("event_time")))
#                  .withColumn("is_weekend", F.col("day_of_week").isin([7,1]))
#                  .withColumn("week_of_month", 
#                              F.weekofyear(F.col("event_date")) - F.weekofyear(F.date_sub(F.col("event_date"), F.dayofmonth(F.col("event_date"))+1))+1)
#                  )

# good_df = (normalised_df.filter((F.col("sales_id").isNotNull()) & (F.col("employee_id").isNotNull()) &
#                                 (F.col("region_id").isNotNull()) & (F.col("quantity").isNotNull()) &
#                                 (F.col("amount").isNotNull()) & (F.col("amount")>=0))
#            .withColumn("processed_time", F.current_timestamp()))

# good_query = (good_df.writeStream
#          .format("delta")
#          .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/sales_chck_slv_v2/")
#          .trigger(availableNow = True)
#          .outputMode("append")
#          .partitionBy("event_date")
#          .toTable("sales_project_streaming.slv.sales")
#          )

# bad_df = (normalised_df.filter((F.col("sales_id").isNull()) | (F.col("employee_id").isNull()) |
#                                 (F.col("region_id").isNull()) | (F.col("quantity").isNull()) |
#                                 (F.col("amount").isNull()) | (F.col("amount")<0))
#            .withColumn("processed_time", F.current_timestamp()))

# bad_query = (bad_df.writeStream
#          .format("delta")
#          .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/bad_sales_chck_slv_v2/")
#          .trigger(availableNow = True)
#          .outputMode("append")
#          .toTable("sales_project_streaming.brz.bad_records_sales")
#          )

# good_query.awaitTermination()
# bad_query.awaitTermination()

In [0]:
# %sql
# optimize sales_project_streaming.slv.sales
# zorder by (region_id);

# vacuum sales_project_streaming.slv.sales;

In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window

# df = spark.readStream.format("delta").table("sales_project_streaming.brz.employees")

# schema = StructType([
#     StructField("employee_id", IntegerType(), False),
#     StructField("before", StructType([
#         StructField("employee_name", StringType(), True),
#         StructField("role", StringType(), True),
#         StructField("department", StringType(), True),
#         StructField("region_id", IntegerType(), True),
#         StructField("joining_date", DateType(), True),
#         StructField("salary", LongType(), True)
#     ]), True),
#     StructField("after", StructType([
#         StructField("employee_name", StringType(), True),
#         StructField("role", StringType(), True),
#         StructField("department", StringType(), True),
#         StructField("region_id", IntegerType(), True),
#         StructField("joining_date", DateType(), True),
#         StructField("salary", LongType(), True)
#     ]), True),
#     StructField("operation", StringType(), True),
#     StructField("event_time", TimestampType(), True)
# ])

# parsed_emp = df.withColumn("parsed", F.from_json(F.col("value"), schema)).select("parsed", "ingestion_ts")

# emp_df = parsed_emp.select("parsed.employee_id", F.col("parsed.before.employee_name").alias("before_employee_name"),
#                    F.col("parsed.before.role").alias("before_role"),
#                    F.col("parsed.before.department").alias("before_department"),
#                    F.col("parsed.before.region_id").alias("before_region_id"),
#                    F.col("parsed.before.joining_date").alias("before_joining_date"),
#                    F.col("parsed.before.salary").alias("before_salary"),
#                    F.col("parsed.after.employee_name").alias("after_employee_name"),
#                    F.col("parsed.after.role").alias("after_role"),
#                    F.col("parsed.after.department").alias("after_department"),
#                    F.col("parsed.after.region_id").alias("after_region_id"),
#                    F.col("parsed.after.joining_date").alias("after_joining_date"),
#                    F.col("parsed.after.salary").alias("after_salary"),
#                    "parsed.operation", "parsed.event_time", F.current_timestamp().alias("processed_time"))

# w = Window.partitionBy("employee_id", "operation").orderBy(F.col("event_time").desc())

# # emp_df = ()

# def scd_imp(batch_df, batch_id):

#     batch_df = (batch_df.withColumn("rn", F.row_number().over(w))
#           .filter(F.col("rn")==1)
#           .drop("rn"))

#     batch_df.createOrReplaceTempView("source_view")

#     spark.sql("""merge into sales_project_streaming.slv.employees t
#               using source_view s
#               on s.employee_id = t.employee_id and t.is_current = True

#                 --- if Deleted - Expire current record and set is_deleted to True

#               when matched and s.operation = "D"
#               then update set t.end_date = to_date(s.event_time),
#                                 t.is_current = False,
#                                 t.is_deleted = True,
#                                 t.processed_time = s.processed_time

#                 --- if updated - Expire curren record and set is_current to False

#                when matched and s.operation = "U"
#                then update set t.end_date = to_date(s.event_time),
#                                 t.is_current = False,
#                                 t.processed_time = s.processed_time;""")
                
#                 # Insert DF for updates and inserts
    
#     insert_df = (batch_df.filter(F.col("operation").isin(["I","U"]))
#                  .select("employee_id", F.col("after_employee_name").alias("employee_name"),
#                          F.col("after_role").alias("role"), F.col("after_department").alias("department"),
#                          F.col("after_region_id").alias("region_id"), F.col("after_joining_date").alias("joining_date"),
#                          F.col("after_salary").alias("salary"), F.to_date(F.col("event_time")).alias("start_date"),
#                          F.lit(None).alias("end_date"), F.lit(True).alias("is_current"),
#                          F.lit(False).alias("is_deleted"), "processed_time"))
    
#     insert_df.write.format("delta").mode("append").saveAsTable("sales_project_streaming.slv.employees")

# query = (emp_df.writeStream.foreachBatch(scd_imp)
#          .option("checkpointLocation", "/Volumes/sales_project_streaming/slv/checkpoints_vol/employees_chck_slv/")
#          .trigger(availableNow = True)
#          .start())

# query.awaitTermination()

In [0]:
# %sql
# select * from sales_project_streaming.slv.employees;

In [0]:
# %sql
# create table sales_project_streaming.slv.employees(
#     employee_sk BIGINT GENERATED ALWAYS AS IDENTITY,
#     employee_id int,
#     employee_name string,
#     role string,
#     department string,
#     region_id int,
#     joining_date date,
#     salary bigint,
#     start_date date,
#     end_date date,
#     is_current boolean,
#     is_deleted boolean,
#     processed_time timestamp
# ) using delta;

In [0]:
%sql
select * from sales_project_streaming.gld.hourly_regional_expenses_agg;